## 음성인식

In [46]:
#!pip install sounddevice numpy scipy
#!pip install git+https://github.com/usefulsensors/moonshine.git
#!pip install ipywidgets
#!pip install -U openai-whisper
#!pip install pyaudio

In [1]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav
import whisper
import threading
import time
import ipywidgets as widgets
from IPython.display import display
import pyaudio

# sounddevice 녹음

#### 오디오 녹음

In [9]:
# 샘플링 레이트 설정
fs = 44100 
audio_data = []
is_recording = True

# 진행 표시바 설정
progress_bar = widgets.IntProgress(min=0, max=100, description='녹음 중...')
progress_label = widgets.Label(value="0.00초 경과")
display(progress_bar, progress_label)

def record_audio(duration):
    global audio_data
    start_time = time.time()  # 시작 시간 기록
    while is_recording and (time.time() - start_time < duration):
        data = sd.rec(int(1 * fs), samplerate=fs, channels=1, dtype='float64')
        sd.wait()
        audio_data.append(data)
        
        elapsed_time = time.time() - start_time
        progress = min(100, int((elapsed_time / duration) * 100))
        progress_bar.value = progress
        progress_label.value = f"{elapsed_time:.2f} 초 경과"

def stop_recording():
    global is_recording
    input("\n종료하려면 'Enter'를 누르세요.")
    is_recording = False
    print("\n녹음이 종료되었습니다.")

max_duration = 60

# 스레드 시작
record_thread = threading.Thread(target=record_audio, args=(max_duration,))
record_thread.start()

stop_thread = threading.Thread(target=stop_recording)
stop_thread.start()

# 스레드가 종료될 때까지 대기
record_thread.join()
stop_thread.join()

IntProgress(value=0, description='녹음 중...')

Label(value='0.00초 경과')


종료하려면 'Enter'를 누르세요. 



녹음이 종료되었습니다.


#### 녹음 데이터 재생

In [10]:
# 녹음된 데이터 재생
print("녹음된 데이터를 재생합니다.")

# 녹음된 데이터 NumPy 배열로 변환
audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# 오디오 데이터 재생
sd.play(audio_data, RATE)
sd.wait()
print("재생이 완료되었습니다.")

녹음된 데이터를 재생합니다.
재생이 완료되었습니다.


# pyaudio 녹음

#### 오디오 녹음

In [23]:
# 녹음 설정
FORMAT = pyaudio.paInt16  # 16비트 정수
CHANNELS = 1  # 모노
#RATE = 44100  # 샘플링 레이트
RATE = 16000  # whisper 요구 샘플링 레이트
CHUNK = 1024  # 청크 크기

# PyAudio 인스턴스 생성
audio = pyaudio.PyAudio()

# 녹음 데이터 저장 변수
frames = []
is_recording = True

# 진행 표시바 설정
progress_bar = widgets.IntProgress(min=0, max=100, description='녹음 중...')
progress_label = widgets.Label(value="0.00초 경과")
display(progress_bar, progress_label)

def record_audio(duration):
    global frames, is_recording
    stream = audio.open(format=FORMAT, channels=CHANNELS,
                        rate=RATE, input=True,
                        frames_per_buffer=CHUNK)
    
    start_time = time.time()  # 시작 시간 기록
    while is_recording and (time.time() - start_time < duration):
        data = stream.read(CHUNK)
        frames.append(data)

        elapsed_time = time.time() - start_time
        progress = min(100, int((elapsed_time / duration) * 100))
        progress_bar.value = progress
        progress_label.value = f"{elapsed_time:.2f} 초 경과"
    
    stream.stop_stream()
    stream.close()

def stop_recording():
    global is_recording
    input("\n종료하려면 'Enter'를 누르세요.")
    is_recording = False
    print("\n녹음이 종료되었습니다.")

# 최대 녹음 시간 (초)
max_duration = 60

# 스레드 시작
record_thread = threading.Thread(target=record_audio, args=(max_duration,))
record_thread.start()

stop_thread = threading.Thread(target=stop_recording)
stop_thread.start()

# 스레드가 종료될 때까지 대기
record_thread.join()
stop_thread.join()

IntProgress(value=0, description='녹음 중...')

Label(value='0.00초 경과')


종료하려면 'Enter'를 누르세요. 



녹음이 종료되었습니다.


#### 녹음 데이터 재생

In [24]:
# 녹음된 데이터 재생
print("녹음된 데이터를 재생합니다.")

# 녹음된 데이터 NumPy 배열로 변환
audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# 오디오 데이터 재생
sd.play(audio_data, RATE)
sd.wait()
print("재생이 완료되었습니다.")

녹음된 데이터를 재생합니다.
재생이 완료되었습니다.


#### 녹음 데이터 전사

In [7]:
# Whisper 모델 로드
model = whisper.load_model("large")

/Users/rynn/anaconda3/lib/python3.11/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


In [25]:
# Whisper로 전사
# 16비트 정수형 데이터를 부동소수점 형식으로 변환
audio_data_float = audio_data.astype(np.float32) / 32768.0 

# Whisper로 전사
# Whisper는 30초 제한이 있으므로, 필요에 따라 슬라이딩 윈도우로 처리할 수 있음
result = model.transcribe(audio_data_float, language=None)
print("전사 결과:", result['text'])

전사 결과:  Did you just... Biosomy?


In [26]:
# Whisper로 전사 (영어, 한국어 따로)
# 16비트 정수형 데이터를 부동소수점 형식으로 변환
audio_data_float = audio_data.astype(np.float32) / 32768.0 

ko_result = model.transcribe(audio_data_float, language="ko", no_speech_threshold=0.3, logprob_threshold=-2.0)
en_result = model.transcribe(audio_data_float, language="en", no_speech_threshold=0.3, logprob_threshold=-2.0)

print("전사 결과:", ko_result["text"])
print("전사 결과:", en_result["text"])

전사 결과:  디디 유 쥬스트 비우사 미
전사 결과:  Did you just... Biosomy?


In [21]:
# Whisper로 전사 (영어, 한국어 따로)
# 16비트 정수형 데이터를 부동소수점 형식으로 변환
audio_data_float = audio_data.astype(np.float32) / 32768.0 

ko_result = model.transcribe(audio_data_float, language="ko", no_speech_threshold=0.3, logprob_threshold=-2.0)
en_result = model.transcribe(audio_data_float, language="en", no_speech_threshold=0.3, logprob_threshold=-2.0)

print("전사 결과:", ko_result["text"])
print("전사 결과:", en_result["text"])

전사 결과:  이것은 컴퓨터입니다.
전사 결과:  This is a computer.
